### Generating artificial data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import os


In [15]:
import numpy as np
import os

np.random.seed(1234)  # For reproducibility
n_series = 2000   # Number of time series per class
n_points = 500    # Number of data points in each time series

#output_dir = 'time_series_data'
#os.makedirs(output_dir, exist_ok=True)

output_dir = os.path.expanduser("~/time_series_data")  # Creates the directory in your home folder
os.makedirs(output_dir, exist_ok=True)


In [16]:
# Function to generate AR, MA, or ARMA data with optional trend and seasonality
def generate_time_series(model_type, order, n_points, trend_strength=0.1, seasonality_amplitude=0.5, seasonality_period=50, include_trend=True):
   if model_type == 'AR':
       params = np.random.uniform(-0.5, 0.5, size=order)
       ar = np.r_[1, -params]
       ma = np.array([1])
   elif model_type == 'MA':
       params = np.random.uniform(-0.5, 0.5, size=order)
       ar = np.array([1])
       ma = np.r_[1, params]
   elif model_type == 'ARMA':
       ar_params = np.random.uniform(-0.5, 0.5, size=order)
       ma_params = np.random.uniform(-0.5, 0.5, size=order)
       ar = np.r_[1, -ar_params]
       ma = np.r_[1, ma_params]
   else:
       raise ValueError("Invalid model type. Use 'AR', 'MA', or 'ARMA'.")
   
   
      # Generate the process
   process = sm.tsa.ArmaProcess(ar, ma)
   data = process.generate_sample(nsample=n_points)
   if include_trend:
       trend = np.linspace(0, trend_strength * n_points, n_points)
       seasonality = seasonality_amplitude * np.sin(2 * np.pi * np.arange(n_points) / seasonality_period)
       data += trend + seasonality
   else:
       seasonality = seasonality_amplitude * np.sin(2 * np.pi * np.arange(n_points) / seasonality_period)
       data += seasonality
   return data

In [17]:
# Loop to generate and save time series plots for each class
model_types = ['AR', 'MA', 'ARMA']
orders = [1, 2, 3]
for model_type in model_types:
   for order in orders:
       for include_trend in [True, False]:
           class_label = f'{model_type}_{order}_with_trend' if include_trend else f'{model_type}_{order}_without_trend'
           class_dir = os.path.join(output_dir, class_label)
           os.makedirs(class_dir, exist_ok=True)
           for i in range(n_series):
               data = generate_time_series(model_type, order, n_points, include_trend=include_trend)
               # Plotting the time series
               plt.figure(figsize=(8, 4))
               plt.plot(data)
               plt.axis('off')  # Turn off axes for a clean image
               plt.savefig(os.path.join(class_dir, f'Series_{i+1}.png'), bbox_inches='tight', pad_inches=0)
               plt.close()
print("Time series generation completed. Time series are saved in the 'time_series_data' directory.")

Time series generation completed. Time series are saved in the 'time_series_data' directory.


# Data splitting 

In [1]:
import os
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import BatchNormalization

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam


/Users/rebeccaganjineh/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
# Parameters
image_size = (244, 244)  # Resize images to this size
batch_size = 32
epochs = 15
main_dir = os.path.expanduser("~/time_series_data")  # Path to your main directory with class subfolders


def load_data(main_dir, image_size):
    data = []
    labels = []
    classes = sorted([cls for cls in os.listdir(main_dir) if os.path.isdir(os.path.join(main_dir, cls))])  # Filter directories only
    class_to_idx = {cls: idx for idx, cls in enumerate(classes)}  # Mapping class names to indices

    for cls in classes:
        class_dir = os.path.join(main_dir, cls)
        for img_file in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_file)
            if img_file.endswith(('.png', '.jpg', '.jpeg')):  # Ensure it's an image file
                img = Image.open(img_path).convert('RGB')  # Convert to RGB
                img = img.resize(image_size)  # Resize image
                data.append(np.array(img))
                labels.append(class_to_idx[cls])

    return np.array(data), np.array(labels), classes






In [6]:
#import os

# Directory containing the dataset
main_dir = os.path.expanduser("~/time_series_data")  # Replace with your actual directory path

# List the class names
classes = sorted([cls for cls in os.listdir(main_dir) if os.path.isdir(os.path.join(main_dir, cls))])

# Print the class names and count
print(f"Classes in dataset: {classes}")
print(f"Number of classes: {len(classes)}")


Classes in dataset: ['ARMA_1_with_trend', 'ARMA_1_without_trend', 'ARMA_2_with_trend', 'ARMA_2_without_trend', 'ARMA_3_with_trend', 'ARMA_3_without_trend', 'AR_1_with_trend', 'AR_1_without_trend', 'AR_2_with_trend', 'AR_2_without_trend', 'AR_3_with_trend', 'AR_3_without_trend', 'MA_1_with_trend', 'MA_1_without_trend', 'MA_2_with_trend', 'MA_2_without_trend', 'MA_3_with_trend', 'MA_3_without_trend']
Number of classes: 18


In [ ]:
# Load images and labels
print("Loading data...")
data, labels, classes = load_data(main_dir, image_size)
data = data / 255.0  # Normalize pixel values to [0, 1]


# First split: 70% training, 30% temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(data, labels, test_size=0.3, random_state=123)

# Second split: 20% validation, 10% test (out of the 30% temp set)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=1/3, random_state=123)

# Convert labels to one-hot encoding
y_train = to_categorical(y_train, num_classes=len(classes))
y_val = to_categorical(y_val, num_classes=len(classes))
y_test = to_categorical(y_test, num_classes=len(classes))



Loading data...


In [7]:
# Define the CNN Model
def build_cnn(input_shape, num_classes):
    model = Sequential([
        Conv2D(64, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        Conv2D(256, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.4),
        
        Conv2D(512, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.5),
        
        Flatten(),
        Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        Dropout(0.5),
        Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')  # Output layer
    ])
    return model

# Parameters
image_size = (224, 224)  # Match the image size in your dataset
input_shape = (image_size[0], image_size[1], 3)
num_classes = len(classes)

# Build the CNN
model = build_cnn(input_shape, num_classes)

# Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


/Users/rebeccaganjineh/Library/Python/3.9/lib/python/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:


# Data Augmentation
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)
datagen.fit(X_train)

# Callbacks
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
early_stopping = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)


/Users/rebeccaganjineh/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


NameError: name 'X_train' is not defined

In [12]:
# Train the model
print("Training model...")
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=batch_size),
    validation_data=(X_val, y_val),
    epochs=50,
    callbacks=[lr_scheduler, early_stopping]
)

Training model...
Epoch 1/50


/Users/rebeccaganjineh/Library/Python/3.9/lib/python/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 of layer "dense" is incompatible with the layer: expected axis -1 of input shape to have value 100352, but received input with shape (None, 16384)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(None, 64, 128, 3), dtype=float32)
  • training=True
  • mask=None

# Building the model

In [7]:
# Build CNN Model
print("Building model...")

# Data Augmentation
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)
datagen.fit(X_train)

# Transfer Learning: Using MobileNetV2
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(image_size[0], image_size[1], 3))
base_model.trainable = False  # Freeze the base model

# Build the model
model = Sequential([
    base_model,
    Flatten(),
    Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
    Dropout(0.5),
    Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
    Dropout(0.5),
    Dense(len(classes), activation='softmax')  # Output layer
])

model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Learning Rate Scheduler and Early Stopping
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
early_stopping = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)



Building model...


In [8]:
# Train the model
print("Training model...")
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=batch_size),
    validation_data=(X_val, y_val),
    epochs=50,  # Increase epochs for transfer learning
    callbacks=[lr_scheduler, early_stopping]
)



Training model...


/Users/rebeccaganjineh/Library/Python/3.9/lib/python/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
209/788 ━━━━━━━━━━━━━━━━━━━━ 1:37 168ms/step - accuracy: 0.0726 - loss: 8.2880

KeyboardInterrupt: 

In [ ]:
# Save the Model
model.save("time_series_classifier.h5")
print("Model saved as 'time_series_classifier.h5'.")